# IDK-1 — Step 4: Pre-training

Train IDK-1 (~100M) dari scratch sampai 100k steps.

**Setup Kaggle — tambahkan dataset sebagai input:**
- Data: `ripkii/idk1-data` (train.bin + val.bin dari notebook 02)
- Tokenizer: `ripkii/idk1-tokenizer`
- Checkpoint (kalau resume): `ripkii/idk1-checkpoint-XXXXX`

**Accelerator:** GPU T4 x2

**Output:** `checkpoints/` → upload ke Kaggle dataset baru setelah session selesai

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import glob
import time
import math
from dataclasses import dataclass

print(f"PyTorch : {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i} : {name} ({vram:.1f} GB)")

## 1. Config

In [ ]:
# ── Model ────────────────────────────────────────────────────────────
@dataclass
class IDK1Config:
    vocab_size  : int   = 40_000
    dim         : int   = 768
    n_layers    : int   = 12
    n_heads     : int   = 12
    n_kv_heads  : int   = 4
    ffn_dim     : int   = 2048
    max_seq_len : int   = 1024
    dropout     : float = 0.0
    norm_eps    : float = 1e-5
    rope_theta  : float = 500_000.0
    logit_cap   : float = 30.0

    @property
    def head_dim(self):
        return self.dim // self.n_heads


# ── Training ─────────────────────────────────────────────────────────
SEQ_LEN    = 512
BATCH_SIZE = 8
GRAD_ACCUM = 8          # effective batch = 64 seq = 32,768 tokens/step
LR         = 3e-4
MIN_LR     = 3e-5
WARMUP     = 1_000
MAX_STEPS  = 30_000
EVAL_EVERY = 500
SAVE_EVERY = 2_500
EVAL_ITERS = 50         # berapa batch untuk estimasi val loss

# ── Paths ─────────────────────────────────────────────────────────────
TRAIN_BIN = "/kaggle/input/datasets/ripkii/idk1-data-new/train.bin"
VAL_BIN   = "/kaggle/input/datasets/ripkii/idk1-data-new/val.bin"
CKPT_DIR  = "/kaggle/working/checkpoints"

# ── Resume ────────────────────────────────────────────────────────────
# Kalau training baru: None
# Kalau resume dari file langsung: path ke .pt file
RESUME_FROM = None  # fresh run dari step 0

os.makedirs(CKPT_DIR, exist_ok=True)
cfg = IDK1Config()

print(f"Effective batch : {BATCH_SIZE * GRAD_ACCUM} seq")
print(f"Tokens/step     : {BATCH_SIZE * GRAD_ACCUM * SEQ_LEN:,}")
print(f"Total tokens    : {BATCH_SIZE * GRAD_ACCUM * SEQ_LEN * MAX_STEPS / 1e9:.2f}B")
print(f"Checkpoint dir  : {CKPT_DIR}")

## 2. Model Architecture

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


def precompute_rope(head_dim, seq_len, theta, device):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t     = torch.arange(seq_len, device=device)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    B, T, H, D = x.shape
    x1  = x[..., :D//2]
    x2  = x[..., D//2:]
    cos = cos[:T].unsqueeze(0).unsqueeze(2)
    sin = sin[:T].unsqueeze(0).unsqueeze(2)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads    = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim   = cfg.head_dim
        self.groups     = cfg.n_heads // cfg.n_kv_heads
        self.wq  = nn.Linear(cfg.dim, cfg.n_heads    * cfg.head_dim, bias=False)
        self.wk  = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wv  = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wo  = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.dim,    bias=False)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads,    self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k = k.repeat_interleave(self.groups, dim=2)
        v = v.repeat_interleave(self.groups, dim=2)
        out = F.scaled_dot_product_attention(
            q.transpose(1,2), k.transpose(1,2), v.transpose(1,2), is_causal=True
        )
        return self.wo(out.transpose(1,2).contiguous().view(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.dim, cfg.ffn_dim, bias=False)
        self.up   = nn.Linear(cfg.dim, cfg.ffn_dim, bias=False)
        self.down = nn.Linear(cfg.ffn_dim, cfg.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.attn      = GroupedQueryAttention(cfg)
        self.ffn_norm  = RMSNorm(cfg.dim, cfg.norm_eps)
        self.ffn       = SwiGLU(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class IDK1Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg     = cfg
        self.embed   = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.layers  = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.dim, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta, device="cpu")
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        x    = self.embed(idx)
        cos  = self.rope_cos[:T]
        sin  = self.rope_sin[:T]
        for layer in self.layers:
            x = layer(x, cos, sin)
        logits = self.lm_head(self.norm(x))
        logits = self.cfg.logit_cap * torch.tanh(logits / self.cfg.logit_cap)
        return logits


print("Architecture OK")

## 3. Data Loader

In [ ]:
class BinaryDataLoader:
    """
    Stream token IDs dari binary file (.bin) secara random.
    Memory-mapped — tidak load semua ke RAM.
    """
    def __init__(self, path, seq_len, batch_size, device):
        self.seq_len    = seq_len
        self.batch_size = batch_size
        self.device     = device
        self.data       = np.memmap(path, dtype=np.uint16, mode="r")
        print(f"Loaded {path}: {len(self.data)/1e9:.2f}B tokens")

    def next_batch(self):
        ix = torch.randint(len(self.data) - self.seq_len - 1, (self.batch_size,))
        x  = torch.stack([torch.from_numpy(self.data[i   : i+self.seq_len  ].astype(np.int64)) for i in ix])
        y  = torch.stack([torch.from_numpy(self.data[i+1 : i+self.seq_len+1].astype(np.int64)) for i in ix])
        return x.to(self.device), y.to(self.device)


train_loader = BinaryDataLoader(TRAIN_BIN, SEQ_LEN, BATCH_SIZE, device)
val_loader   = BinaryDataLoader(VAL_BIN,   SEQ_LEN, BATCH_SIZE, device)

# Quick test
xb, yb = train_loader.next_batch()
print(f"Batch shape : x={xb.shape}, y={yb.shape}")

## 4. Init Model + Optimizer

In [ ]:
model = IDK1Model(cfg)

# Multi-GPU kalau ada
if torch.cuda.device_count() > 1:
    print(f"DataParallel: {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model = model.to(device)

# Hitung params
raw_model  = model.module if isinstance(model, nn.DataParallel) else model
total_params = sum(p.numel() for p in raw_model.parameters())
print(f"Params: {total_params/1e6:.2f}M")

# AdamW — weight decay hanya untuk weight tensors, bukan bias/norm
decay_params    = [p for n, p in raw_model.named_parameters() if p.dim() >= 2]
no_decay_params = [p for n, p in raw_model.named_parameters() if p.dim() < 2]

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params,    "weight_decay": 0.1},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LR,
    betas=(0.9, 0.95),
    eps=1e-8,
    fused=True if device == "cuda" else False,
)

scaler = torch.cuda.amp.GradScaler()  # untuk float16 training

print("Model + optimizer OK")

## 5. Resume dari Checkpoint (kalau ada)

In [ ]:
step          = 0
best_val_loss = float("inf")

if RESUME_FROM:
    # Support langsung path ke file .pt
    if RESUME_FROM.endswith(".pt") and os.path.isfile(RESUME_FROM):
        resume_path = RESUME_FROM
    else:
        # Fallback: cari step_*.pt tertinggi di folder
        pts = []
        for f in glob.glob(os.path.join(RESUME_FROM, "step_*.pt")):
            try:
                s = int(os.path.basename(f).replace("step_", "").replace(".pt", ""))
                pts.append((s, f))
            except:
                pass
        resume_path = max(pts, key=lambda x: x[0])[1] if pts else None

    if resume_path:
        print(f"Resuming dari: {resume_path}")
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)

        state_dict = ckpt["model"]
        if any(k.startswith("module.") for k in state_dict):
            state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        raw_model.load_state_dict(state_dict)

        optimizer.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        step          = ckpt["step"]
        best_val_loss = ckpt.get("best_val_loss", float("inf"))

        print(f"Resumed: step={step}, best_val_loss={best_val_loss:.4f}")
    else:
        print(f"Tidak ada checkpoint di {RESUME_FROM}, mulai dari awal.")
else:
    print("Training dari awal (step 0)")

## 6. Learning Rate Schedule

In [ ]:
def get_lr(step):
    """Cosine decay dengan linear warmup."""
    # Warmup
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    # Setelah MAX_STEPS → MIN_LR
    if step >= MAX_STEPS:
        return MIN_LR
    # Cosine decay
    progress = (step - WARMUP) / (MAX_STEPS - WARMUP)
    coeff    = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + coeff * (LR - MIN_LR)


# Quick check schedule
for s in [0, 100, 500, 1000, 5000, 50000, 100000]:
    print(f"  step {s:>6} → lr = {get_lr(s):.2e}")

## 7. Eval Function

In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split, loader in [("train", train_loader), ("val", val_loader)]:
        batch_losses = []
        for _ in range(EVAL_ITERS):
            x, y = loader.next_batch()
            with torch.cuda.amp.autocast(dtype=torch.float16):
                logits = model(x)
                loss   = F.cross_entropy(logits.view(-1, cfg.vocab_size), y.view(-1))
            batch_losses.append(loss.item())
        losses[split] = sum(batch_losses) / len(batch_losses)
    model.train()
    return losses


print("Eval function OK")

## 8. Save Checkpoint

In [ ]:
def save_checkpoint(step, val_loss, is_best=False):
    state = {
        "step"          : step,
        "model"         : raw_model.state_dict(),
        "optimizer"     : optimizer.state_dict(),
        "scaler"        : scaler.state_dict(),
        "best_val_loss" : best_val_loss,
        "cfg"           : cfg.__dict__,
    }
    # Simpan dengan nama step yang jelas
    path = os.path.join(CKPT_DIR, f"step_{step:06d}.pt")
    torch.save(state, path)

    # Timpa latest.pt
    latest = os.path.join(CKPT_DIR, "latest.pt")
    torch.save(state, latest)

    # Kalau val_loss terbaik, timpa best.pt juga
    if is_best:
        best = os.path.join(CKPT_DIR, "best.pt")
        torch.save(state, best)
        print(f"  ★ New best! val_loss={val_loss:.4f} → saved best.pt")

    return path


print("Save function OK")

## 9. Training Loop

In [ ]:
model.train()
optimizer.zero_grad()

t_start        = time.time()
log_loss       = 0.0
LOG_EVERY      = 50
step_start     = step
losses         = {}

print(f"Training IDK-1 — start dari step {step}")
print(f"Target: {MAX_STEPS:,} steps")
print("-" * 60)

while step < MAX_STEPS:

    # ── Set LR ──────────────────────────────────────────────────────
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    # ── Eval ────────────────────────────────────────────────────────
    if step % EVAL_EVERY == 0:
        losses   = estimate_loss()
        is_best  = losses["val"] < best_val_loss
        if is_best:
            best_val_loss = losses["val"]
        elapsed  = time.time() - t_start
        print(f"step {step:>6} | train={losses['train']:.4f} | val={losses['val']:.4f} | lr={lr:.2e} | t={elapsed:.0f}s")

    # ── Save ────────────────────────────────────────────────────────
    if step % SAVE_EVERY == 0 and step > 0:
        # SAVE_EVERY adalah kelipatan EVAL_EVERY, jadi losses selalu tersedia
        is_best_save = losses.get("val", float("inf")) == best_val_loss
        path = save_checkpoint(step, best_val_loss, is_best=is_best_save)
        print(f"  Saved: {os.path.basename(path)}")

    # ── Gradient Accumulation ───────────────────────────────────────
    accum_loss = 0.0
    for micro_step in range(GRAD_ACCUM):
        x, y = train_loader.next_batch()
        with torch.cuda.amp.autocast(dtype=torch.float16):
            logits = model(x)
            loss   = F.cross_entropy(logits.view(-1, cfg.vocab_size), y.view(-1))
            loss   = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    # Gradient clipping — cegah gradient explode
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    log_loss += accum_loss
    step     += 1

    # ── Log progress ────────────────────────────────────────────────
    if step % LOG_EVERY == 0:
        avg_loss   = log_loss / LOG_EVERY
        elapsed    = time.time() - t_start
        steps_done = step - step_start
        tok_s      = steps_done * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN / elapsed
        eta_h      = (MAX_STEPS - step) * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN / tok_s / 3600
        print(f"  step {step:>6} | loss={avg_loss:.4f} | {tok_s/1000:.1f}k tok/s | ETA {eta_h:.1f}h")
        log_loss = 0.0


# ── Save checkpoint akhir ──────────────────────────────────────────
losses = estimate_loss()
save_checkpoint(step, losses["val"], is_best=losses["val"] < best_val_loss)
print("\n" + "=" * 60)
print(f"TRAINING SELESAI — step {step}")
print(f"Final val loss : {losses['val']:.4f}")
print(f"Best val loss  : {best_val_loss:.4f}")
print(f"Checkpoint dir : {CKPT_DIR}")
print("=" * 60)

## 10. Next Step

Setelah training selesai atau session habis:

1. Download folder `checkpoints/` dari `/kaggle/working/`
2. Upload ke Kaggle dataset baru: `idk1-checkpoint-XXXXX` (isi XXXXX dengan step terakhir)
3. Session berikutnya: set `RESUME_FROM` ke path dataset tersebut
4. Kalau sudah 100k steps → lanjut ke `05_eval.ipynb`

In [ ]:
# Cek isi checkpoint folder
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pt")))
print(f"Checkpoints tersimpan ({len(ckpts)} file):")
for c in ckpts:
    size_mb = os.path.getsize(c) / 1e6
    print(f"  {os.path.basename(c):30s} {size_mb:.0f} MB")